In [ ]:
import pandas as pd

# --- Pobranie danych z pliku .csv i dopasowanie typu danych dla daty --- #

revenues_per_day = pd.read_csv("revenues_per_day.csv")

# Podstawowa Weryfikacja danych --- #
revenues_per_day.info()
revenues_per_day.describe()
print(revenues_per_day.head())

# Ustawiam odpowiednie typy danych
import pandas as pd
revenues_per_day['date'] = pd.to_datetime(revenues_per_day['date'])

In [ ]:
# --- Ograniczam dane --- #
# Z uwagi na ograniczenia API (1000 zapytań), analizę ograniczę do TOP 300 filmów wg przychodów

top_300_movies = revenues_per_day.copy()

# Pozbywam się znaków specjalnych takich jak ':' oraz '-', aby skuteczniej szukać filmów
top_300_movies['title_cleaned'] = top_300_movies['title'].str.replace(':',' ').str.replace('-',' ')

# Usuwam nadmierne spacje na zewnątrz i w środku nazw filmów
top_300_movies['title_cleaned'] = top_300_movies['title_cleaned'].str.strip()
top_300_movies['title_cleaned'] = top_300_movies['title_cleaned'].str.replace(r'\s+', ' ', regex=True)
top_300_movies = top_300_movies.groupby(
    by='title_cleaned'
).agg(
    {'revenue':'sum'}
).sort_values(
    by='revenue',
    ascending=False
).iloc[:300]
unique_movies = top_300_movies.index.to_list()

In [ ]:
# --- Zapytanie do API ---

import requests
import pandas as pd
import time

# Klucz API do wygenerowania w przypadku potrzeby uruchomienia skryptu
API_KEY = "a4265a6e" # Do podmiany w przypadku chęci uruchomienia
HTTP = "https://www.omdbapi.com/"
list_of_movies = []

print("Rozpoczynam pobieranie danych.")

try:
  with requests.Session() as s:

    # Ustawiam parametry sesji
    s.params.update({"apikey":API_KEY})

    # Wykonuję zapytania dla każdego filmu na liście
    for movie_title in unique_movies:
      print(f"Wyszukuję {movie_title}")
      r = s.get(url=HTTP, params={"s":movie_title})
      print(r.url)
      # Sprawdzenie błędów 404
      r.raise_for_status()
      data_general = r.json()
      if data_general.get("Response") == 'True':
        search_result = data_general.get("Search")
        if search_result:
          movie_id = search_result[0].get("imdbID")
          if movie_id:
            # Używam ID aby pobrać pełne dane o filmie
            detailed_response = s.get(url=HTTP, params={"i":movie_id})  
            detailed_response.raise_for_status()
            data_detailed = detailed_response.json()
            list_of_movies.append(data_detailed)
          else:
            print(f"Dla filmu {movie_title} nie znaleziono ID")
      else:
        print(f"Nie znaleziono filmu: {movie_title}")
      # Ustawiam chwilę przerwy po każdym zapytaniu, aby nie obciążać strony
      time.sleep(0.5)
# Obsługa błędów
except requests.exceptions.RequestException as e:
    print(f"\nWystąpił błąd sieciowy lub HTTP: {e}")
except KeyError as e:
    print(f"\nWystąpił błąd przetwarzania danych (prawdopodobnie zła struktura JSON): {e}")
except Exception as e:
    print(f"\nWystąpił nieoczekiwany błąd: {e}")
print("Zakończono pobieranie danych")

if list_of_movies:
  df = pd.DataFrame(data=list_of_movies)
  print("Utworzono DataFrame na bazie pobranych danych.")
else:
  print("Nie udało się pobrać żadnych danych do utworzenia DataFrame.")

In [ ]:
import pandas as pd
import numpy as np

# --- Budowa tabeli z filmami ---
# Ograniczenie kolumn
columns_to_choose = ['imdbID', 'Title', 'Year', 'Rated', 'Released', 'Runtime', 'Genre', 'Director', 'Metascore', 'imdbRating', 'imdbVotes', 'BoxOffice']
dim_movies = df[columns_to_choose]

# Weryfikacja pusty wartości w kolumnach
# Okazuje się, że nulli nie ma ale są zwroty typu 'N/A'
dim_movies.replace('N/A', np.nan, inplace=True)
print("Puste wartości w dim_movies:")
print(dim_movies.isnull().sum())

# Oczyszczenie danych z tabeli wymiarów
dim_movies['Rated'].replace(['N/A', 'Not Rated'], 'Unknown', inplace=True)
dim_movies['Rutime'] = dim_movies['Runtime'].str.replace(' min', '').astype(float)
dim_movies['Rutime'].fillna(0, inplace=True)
dim_movies['Rutime'] = dim_movies['Rutime'].astype(int)
dim_movies['Director'] = dim_movies['Director'].str.split(',').str[0]
dim_movies['Director'].fillna('Unknown', inplace=True)
dim_movies['Metascore'] = pd.to_numeric(dim_movies['Metascore'], errors='coerce').fillna(0).astype(int)
dim_movies['imdbRating'] = pd.to_numeric(dim_movies['imdbRating'], errors='coerce').fillna(0.0)
dim_movies['BoxOffice'] = dim_movies['BoxOffice'].replace({r'[$,]': ''}, regex=True)
dim_movies['BoxOffice'] = pd.to_numeric(dim_movies['BoxOffice'], errors='coerce').fillna(0).astype(int)
dim_movies['imdbVotes'] = dim_movies['imdbVotes'].replace(',', '', regex=True)
dim_movies['imdbVotes'] = pd.to_numeric(dim_movies['imdbVotes'], errors='coerce').fillna(0).astype(int)
dim_movies['Year'] = dim_movies['Year'].str.extract(r'(\d+)')
dim_movies = dim_movies.drop_duplicates(subset=['Title'], keep='first')
dim_movies.insert(0, 'movie_id', range(1, 1 + len(dim_movies)))

# Tabela wymiarów filmów
# --- Budowa tabeli z datami ---
# Weryfikuję minimalną i maksymalną datę w analizowanym zbiorze (szerokim)

revenues_per_day['date'] = pd.to_datetime(revenues_per_day['date'])
min_date = revenues_per_day['date'].min()
max_date = revenues_per_day['date'].max()

# Tworzę obiekt DateTimeIndex na bazie daty minimalnej i maksymalnej
all_dates = pd.date_range(start=min_date, end=max_date, freq='D')

# Tworzę tabelę z wymiarami dat
dim_date = pd.DataFrame({'date':all_dates})
dim_date['year'] = dim_date['date'].dt.year
dim_date['month'] = dim_date['date'].dt.month
dim_date['month_name'] = dim_date['date'].dt.month_name()
dim_date['day'] = dim_date['date'].dt.day
dim_date['quarter'] = dim_date['date'].dt.quarter
dim_date['day_name'] = dim_date['date'].dt.day_name()
dim_date['day_of_week'] = dim_date['date'].dt.dayofweek

# Dodaję klucz do połączenia danych
dim_date.insert(0, 'date_id', range(1, 1 + len(dim_date)))

# Konwertuję klucze na tekst, aby nie było problemów przy konwersji przez ODBC
dim_date['date_id'] = dim_date['date_id'].astype(str)

# Tabela wymiarów data
print(dim_date.head(5))

In [ ]:
# --- Budowa tabeli faktów ---
# Najpierw łączę się z wymiarem filmów
fact_table_temp = pd.merge(
    revenues_per_day,
    dim_movies[['movie_id', 'Title']],
    left_on='title',
    right_on='Title'
)

# Potem łączę się z wymiarem dat
fact_table_final = pd.merge(
    fact_table_temp,
    dim_date[['date_id', 'date']],
    left_on='date',
    right_on='date'
)

print(fact_table_final.head())

# Na końcu odfiltrowuje zbędne kolumny z ostatniego DataFrame
columns_to_choose = ['movie_id', 'date_id', 'revenue', 'theaters']

# Tabela faktów
fact_table = fact_table_final[columns_to_choose]

# Oczyszczam tabelę faktów
fact_table['revenue'].replace('N/A', np.nan, inplace=True)
fact_table['theaters'] = fact_table['theaters'].replace(',', '', regex=True)
fact_table['theaters'] = pd.to_numeric(fact_table['theaters'], errors='coerce').fillna(0).astype(int)

# Buduję miarę średniej wartości przychodów per film
fact_table['sum_revenue'] = fact_table.groupby(
    by='movie_id'
)['revenue'].transform('sum')


In [ ]:
# import danych na bazę
from sqlalchemy import create_engine

# Tworzymy silnik połączenia z bazą
engine = create_engine('sqlite:///movies.db')

# Zapisz każdy DataFrame jako osobną tabelę w bazie
dim_movies.to_sql('Dim_Movie', con=engine, if_exists='replace', index=False)
dim_date.to_sql('Dim_Date', con=engine, if_exists='replace', index=False)
fact_table.to_sql('Fact_MovieRevenue', con=engine, if_exists='replace', index=False)

print("Model danych został pomyślnie załadowany do pliku 'movies.db'!")
